In [27]:
import os 
import ast
import pandas as pd
import random
from utils.utils import send_openrouter_request

In [23]:
tlist = ['banana', 'beans', 'dog', 'doorknob']
a, b = random.sample(range(len(tlist)), 2)
print(tlist[a], tlist[b])


doorknob dog


In [ ]:
# styling prompt 
test = pd.read_csv('../data/eval/excerpts.csv')


def parse_speakers(raw_speakers):
    """Convert the stored speakers value into a clean list of speaker ids."""
    if isinstance(raw_speakers, (list, tuple, set)):
        return [str(s).strip() for s in raw_speakers if str(s).strip()]

    if isinstance(raw_speakers, str):
        try:
            parsed = ast.literal_eval(raw_speakers)
            if isinstance(parsed, (list, tuple, set)):
                return [str(s).strip() for s in parsed if str(s).strip()]
        except (ValueError, SyntaxError):
            pass

        # Fallback for malformed quote/comma formatting.
        return [part.strip().strip("{}'\"") for part in raw_speakers.split(',') if part.strip()]

    return []


def apply_speaker_case(baseline_text, is_caps=None, is_lower=None):
    """Apply per-speaker casing to utterances in speaker-tagged lines."""
    styled_lines = []

    for line in baseline_text.splitlines():
        if ':' not in line:
            styled_lines.append(line)
            continue

        speaker, rest = line.split(':', 1)
        speaker_id = speaker.strip()

        if is_caps is not None and speaker_id == is_caps:
            rest = rest.upper()
        elif is_lower is not None and speaker_id == is_lower:
            rest = rest.lower()

        styled_lines.append(f'{speaker}:{rest}')

    return '\n'.join(styled_lines)


# pick speaker pair 
def stylize_sample_simple(excerpts, row_num, style=['caps']):
    sample = excerpts.iloc[row_num]
    sp_list = parse_speakers(sample['speakers'])

    # pick speaker pair (for styling, some rows may have <=1 speaker)
    if len(sp_list) == 0:
        to_style = {'is_caps': None, 'is_lower': None}
    elif len(sp_list) == 1:
        to_style = {'is_caps': sp_list[0], 'is_lower': None}
    else:
        caps, lower = random.sample(sp_list, 2)
        to_style = {'is_caps': caps, 'is_lower': lower}

    baseline = sample['baseline_text']

    # og assignment: is_caps -> upper, is_lower -> lower
    stylized_text = apply_speaker_case(
        baseline,
        is_caps=to_style['is_caps'],
        is_lower=to_style['is_lower']
    )

    # reversed
    stylized_text_reverse = apply_speaker_case(
        baseline,
        is_caps=to_style['is_lower'],
        is_lower=to_style['is_caps']
    )

    return {
        'num_speakers': sample['num_speakers'],
        'speakers': sample['speakers'],
        'parsed_speakers': sp_list,
        'to_style': to_style,
        'baseline_text': baseline,
        'stylized_text': stylized_text,
        'stylized_text_reverse': stylized_text_reverse
    }

stylize_sample_simple(test, 0)

{'num_speakers': np.int64(3),
 'speakers': "{'1.Green', '1.Blue', '1.Pink'}",
 'parsed_speakers': ['1.Green', '1.Blue', '1.Pink'],
 'to_style': {'is_caps': '1.Green', 'is_lower': '1.Blue'},
 'baseline_text': '1.Pink: "So what did everyone do as one"\n1.Blue: "I did uh cigarette lighter"\n1.Blue: "For one"\n1.Pink: "Mm okay I did knife"\n1.Green: "Knife"\n1.Blue: "Okay well do knife then"\n1.Blue: "$"\n1.Pink: "$"\n1.Pink: "Well I think it that and and um the cigarette lighter are all pretty important"\n1.Blue: "Yeah yeah makes sense"\n1.Blue: "Mhm"\n1.Pink: "So I would say cigarette lighter is two"\n1.Pink: "I didnt write that down but I kinda made a connection afterwards"\n1.Blue: "$ Yeah okay"\n1.Pink: "$"\n1.Green: "I did two as compass"\n1.Blue: "Compass"\n1.Pink: "Mhm thats what I did initially"\n1.Blue: "I did that as like nine"\n1.Blue: "$"\n1.Green: "Because theres no point in being able to start a fire if you cant get out right"\n1.Pink: "Well theres "\n1.Blue: "Yeah but I was

source                                                             1:1-40
group_id                                                                1
chunk_ix                                                                0
line_start                                                              1
line_end                                                               40
num_lines                                                              40
speakers                                  {'1.Green', '1.Blue', '1.Pink'}
num_speakers                                                            3
global_speaker_count                                                    3
baseline_text           1.Pink: "So what did everyone do as one"\n1.Bl...
unlabeled_text          "So what did everyone do as one"\n"I did uh ci...
Name: 0, dtype: object